In [7]:
def load_data():
    import pandas as pd
    import numpy as np
    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    # now we split the data into data and target
    data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    target = raw_df.values[1::2, 2]
    # These are feature names
    feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
    # create a data frame
    df = pd.DataFrame(data, columns=feature_names)
    df['MEDV'] = target # here MEDV is our target variable
    
    return df

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Prabhakar\AppData\Local\Temp\ipykernel_82824\1283889103.py:5: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


In [22]:
df = load_data()
print(df.head())

      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31   0.0  0.538  6.575  65.2  4.0900  1.0  296.0   
1  0.02731   0.0   7.07   0.0  0.469  6.421  78.9  4.9671  2.0  242.0   
2  0.02729   0.0   7.07   0.0  0.469  7.185  61.1  4.9671  2.0  242.0   
3  0.03237   0.0   2.18   0.0  0.458  6.998  45.8  6.0622  3.0  222.0   
4  0.06905   0.0   2.18   0.0  0.458  7.147  54.2  6.0622  3.0  222.0   

   PTRATIO       B  LSTAT  MEDV  
0     15.3  396.90   4.98  24.0  
1     17.8  396.90   9.14  21.6  
2     17.8  392.83   4.03  34.7  
3     18.7  394.63   2.94  33.4  
4     18.7  396.90   5.33  36.2  


In [36]:
def evaluate_models(param_name, param_values):
    """
    Tune one RandomForest hyperparameter across 3 values and print train/test metrics.

    Example:
        evaluate_models("max_depth", [None, 5, 10])
        evaluate_models("n_estimators", [100, 200, 500])
        evaluate_models("max_features", ["auto", "sqrt", 0.8])
    """
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.ensemble import RandomForestRegressor

    assert len(param_values) == 3, "Provide exactly 3 values for the hyperparameter."

    df = load_data()
    X = df.drop(columns=['MEDV']).values
    y = df['MEDV'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = []
    for val in param_values:
        # Base model (you can change other defaults if you wish)
        model = RandomForestRegressor(
            n_estimators=200,  # default; will be overridden if param_name == "n_estimators"
            random_state=42,
            n_jobs=-1
        )
        # Override the chosen hyperparameter
        model.set_params(**{param_name: val})

        model.fit(X_train, y_train)

        # Train metrics
        y_pred_tr = model.predict(X_train)
        tr_mse = mean_squared_error(y_train, y_pred_tr)
        tr_r2 = r2_score(y_train, y_pred_tr)

        # Test metrics
        y_pred_te = model.predict(X_test)
        te_mse = mean_squared_error(y_test, y_pred_te)
        te_r2 = r2_score(y_test, y_pred_te)

        # Pretty value for printing (handle None)
        val_print = "None" if val is None else str(val)
        results.append((val_print, tr_mse, tr_r2, te_mse, te_r2))

    # Print table
    print(f"RandomForest (tuning '{param_name}') – Training vs Testing Performance")
    print("{:<15} {:>12} {:>12} {:>12} {:>12}".format(param_name, "Train MSE", "Train R²", "Test MSE", "Test R²"))
    for v, tr_mse, tr_r2, te_mse, te_r2 in results:
        print("{:<15} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f}".format(v, tr_mse, tr_r2, te_mse, te_r2))

if __name__ == "__main__":
    # EXAMPLES (uncomment one at a time)
    evaluate_models("max_depth", [None, 5, 10])
    # evaluate_models("n_estimators", [100, 200, 500])
    # evaluate_models("min_samples_split", [2, 5, 10]) 
   

RandomForest (tuning 'max_depth') – Training vs Testing Performance
max_depth          Train MSE     Train R²     Test MSE      Test R²
None                   1.949        0.978        8.510        0.884
5                      5.692        0.934        9.501        0.870
10                     2.235        0.974        8.474        0.884
